In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

In [3]:
# Load data
df = pd.read_csv('gas_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

In [5]:
# Feature engineering - create time-based features
df['hour'] = df['Timestamp'].dt.hour
df['day_part'] = pd.cut(df['hour'], 
                       bins=[0, 6, 12, 18, 24],
                       labels=['night', 'morning', 'afternoon', 'evening'])

In [41]:
# Calculate  overall saturation level (you may need to adjust the threshold)
def classify_gas(ppm):
    if ppm < 10: return 'Low'
    elif ppm < 20: return 'Moderate'
    elif ppm < 30: return 'High'
    else: return 'Critical'

# Apply to each gas column
gas_columns = ['Ammonia', 'Benzene', 'CNG', 'CO', 'Hydrogen', 'LPG', 'Smoke']
for gas in gas_columns:
    df[f'{gas}_level'] = df[gas].apply(classify_gas)

In [51]:
from collections import Counter

def get_most_common(row):
    counts = Counter(row)
    return counts.most_common(1)[0][0]  # Returns the most frequent level

df['saturation'] = df[level_columns].apply(get_most_common, axis=1)

print(df[['Timestamp'] + level_columns + ['saturation']].head())

                         Timestamp Ammonia_level Benzene_level CNG_level  \
0 2025-03-23 23:02:49.251000+00:00      Moderate           Low      High   
1 2025-03-23 23:03:12.058000+00:00      Moderate           Low      High   
2 2025-03-23 23:03:17.384000+00:00      Moderate           Low      High   
3 2025-03-23 23:03:33.084000+00:00      Moderate           Low      High   
4 2025-03-23 23:03:38.380000+00:00      Moderate           Low      High   

   CO_level Hydrogen_level LPG_level Smoke_level saturation  
0  Moderate            Low  Moderate         Low   Moderate  
1  Moderate            Low  Moderate         Low   Moderate  
2  Moderate            Low  Moderate         Low   Moderate  
3  Moderate            Low  Moderate         Low   Moderate  
4  Moderate            Low  Moderate         Low   Moderate  


In [52]:
# Prepare features and target
features = ['Ammonia', 'Benzene', 'CNG', 'CO', 'Hydrogen', 'LPG', 'Smoke', 'hour']
X = df[features]
y = df['tomorrow_saturation']

In [53]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [54]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Critical       1.00      0.98      0.99        44
        High       0.96      1.00      0.98        24

    accuracy                           0.99        68
   macro avg       0.98      0.99      0.98        68
weighted avg       0.99      0.99      0.99        68



In [55]:
# Feature importance
importances = model.feature_importances_
for feature, importance in zip(features, importances):
    print(f"{feature}: {importance:.4f}")

Ammonia: 0.1351
Benzene: 0.1132
CNG: 0.1407
CO: 0.1051
Hydrogen: 0.1221
LPG: 0.1365
Smoke: 0.0538
hour: 0.1936


In [56]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import LabelEncoder

In [ ]:
# 1. Fix the SettingWithCopyWarning
# Make sure you're working with a proper DataFrame copy if needed
df = df.copy()  # This ensures we're not working on a view

# 2. Convert saturation levels to numerical values
le = LabelEncoder()
df.loc[:, 'saturation_num'] = le.fit_transform(df['current_saturation'])  # Using .loc to avoid warning

# 3. Create time series with proper frequency
ts = df.set_index('Timestamp')['saturation_num']

# Add frequency information (critical for ARIMA)
# Assuming your data is hourly:
ts = ts.asfreq('H')  # 'H' for hourly, adjust if different

# 4. Fit ARIMA model with proper parameter tuning
# Start with simpler parameters (24 might be too large for your dataset)
try:
    model = ARIMA(ts, order=(3,1,1))  # Simpler (p,d,q) parameters
    model_fit = model.fit()
    
    # 5. Forecast next day
    forecast = model_fit.forecast(steps=24)
    forecast_levels = le.inverse_transform(forecast.round().astype(int))
    
    print("Next day's saturation predictions:")
    print(pd.Series(forecast_levels, index=pd.date_range(start=ts.index[-1] + pd.Timedelta(hours=1), periods=24, freq='H')))
    
except Exception as e:
    print(f"Model fitting failed: {str(e)}")
    print("Possible issues:")
    print("- Not enough data points (need at least 50-100 observations)")
    print("- Try simpler ARIMA parameters like (1,1,1))")
    print("- Check for missing values in time series)")

Next day's saturation predictions:
2025-03-24 04:02:49.251000+00:00    High
2025-03-24 05:02:49.251000+00:00    High
2025-03-24 06:02:49.251000+00:00    High
2025-03-24 07:02:49.251000+00:00    High
2025-03-24 08:02:49.251000+00:00    High
2025-03-24 09:02:49.251000+00:00    High
2025-03-24 10:02:49.251000+00:00    High
2025-03-24 11:02:49.251000+00:00    High
2025-03-24 12:02:49.251000+00:00    High
2025-03-24 13:02:49.251000+00:00    High
2025-03-24 14:02:49.251000+00:00    High
2025-03-24 15:02:49.251000+00:00    High
2025-03-24 16:02:49.251000+00:00    High
2025-03-24 17:02:49.251000+00:00    High
2025-03-24 18:02:49.251000+00:00    High
2025-03-24 19:02:49.251000+00:00    High
2025-03-24 20:02:49.251000+00:00    High
2025-03-24 21:02:49.251000+00:00    High
2025-03-24 22:02:49.251000+00:00    High
2025-03-24 23:02:49.251000+00:00    High
2025-03-25 00:02:49.251000+00:00    High
2025-03-25 01:02:49.251000+00:00    High
2025-03-25 02:02:49.251000+00:00    High
2025-03-25 03:02:49.25

C:\Users\USER\AppData\Local\Temp\ipykernel_15860\3799914275.py:14: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts = ts.asfreq('H')  # 'H' for hourly, adjust if different
c:\Users\USER\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
C:\Users\USER\AppData\Local\Temp\ipykernel_15860\3799914275.py:27: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  print(pd.Series(forecast_levels, index=pd.date_range(start=ts.index[-1] + pd.Timedelta(hours=1), periods=24, freq='H')))


: 